# Example 7.8 / 1: Change One Setting at a Time

Companion notebook for **7.8 Branching Priorities and Solver Tuning** in the AMPL lab Documentation, section "Change One Setting at a Time". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Change One Setting at a Time

Every HiGHS solve message ends with the work done: simplex iterations and branch-and-bound nodes.
To judge an option, solve the same instance with and without it and compare those counts along
with the objective. Here the option `mip:pscostreliability`, which controls how much branching
history HiGHS collects before trusting it, is set to 0:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                 # items that can be packed
param value {ITEMS} > 0;   # value of each item
param weight {ITEMS} > 0;  # weight of each item
param volume {ITEMS} > 0;  # volume of each item
param max_weight > 0;      # weight capacity
param max_volume > 0;      # volume capacity
var Take {ITEMS} binary;   # 1 if the item is packed
maximize Total_Value: sum {i in ITEMS} value[i] * Take[i];  # total value packed
subject to Weight: sum {i in ITEMS} weight[i] * Take[i] <= max_weight;  # weight capacity
subject to Volume: sum {i in ITEMS} volume[i] * Take[i] <= max_volume;  # volume capacity

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
items = ["i1", "i2", "i3", "i4", "i5", "i6", "i7", "i8"]
ampl.set["ITEMS"] = items
ampl.param["value"] = dict(zip(items, [25, 47, 44, 18, 33, 48, 40, 50]))
ampl.param["weight"] = dict(zip(items, [23, 7, 24, 5, 20, 13, 22, 12]))
ampl.param["volume"] = dict(zip(items, [11, 27, 20, 22, 22, 20, 17, 25]))
ampl.param["max_weight"] = 63
ampl.param["max_volume"] = 82

In [5]:
# run.py
for options in ["", "mip:pscostreliability=0"]:
    ampl.solve(solver="highs", highs_options=options, verbose=False)
    message = ampl.get_value("solve_message").splitlines()
    print(options or "defaults", "|", ampl.solve_result, ampl.obj["Total_Value"].value(),
          "|", message[-2], "|", message[-1])

defaults | solved 153.0 | 68 simplex iterations | 6 branching nodes
mip:pscostreliability=0 | solved 153.0 | 45 simplex iterations | 16 branching nodes


### Pure AMPL: data.dat and model.run

`ampl.get_value("solve_message")` returns the message the solver printed. Both runs prove the
same optimum, 153, but the changed setting needs more nodes, so the default stays. On a real model
compare several representative instances, not one: Comparing Formulations
Computationally (AMPL lab documentation, section 7.10.2) describes how to set up and record such a comparison.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: ITEMS: value weight volume :=
  i1  25  23  11
  i2  47   7  27
  i3  44  24  20
  i4  18   5  22
  i5  33  20  22
  i6  48  13  20
  i7  40  22  17
  i8  50  12  25 ;
param max_weight := 63;
param max_volume := 82;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display solve_message;
option highs_options 'mip:pscostreliability=0';
solve;
display solve_message;

solve_message = 'HiGHS 1.14.0: optimal solution; objective 153\
68 simplex iterations\
6 branching nodes\
'

HiGHS 1.14.0:   mip:pscostreliability = 0
solve_message = 'HiGHS 1.14.0: optimal solution; objective 153\
45 simplex iterations\
16 branching nodes\
'

